# Distributed Linear Regression with PySpark: Predicting Restaurant Tips

## 1. System Initialization & SparkSession Architecture

### Code Explanation: PySpark Execution Architecture
In distributed machine learning environments, Apache Spark employs a decoupled master-worker computational model:
- **Driver Program**: Coordinates execution, analyzes user code, maintains the Directed Acyclic Graph (DAG) of transformations, invokes the Catalyst optimizer for query optimization, and schedules tasks via the Tungsten physical execution engine.
- **Executors**: Worker processes executing parallel tasks across isolated partitions, retaining cached data in-memory or spilling to disk.

The `SparkSession` (standardized in Spark 2.0+) acts as the single unified entry point for programming Spark with DataFrame and Dataset APIs. It subsumes historical discrete contexts (`SparkContext`, `SQLContext`, `HiveContext`) into a cohesive interface.

In this local containerized Docker deployment, `SparkSession.builder.appName(...).getOrCreate()` instantiates an isolated local execution context bridging Python and the underlying JVM via Py4J.


In [1]:
from pyspark.sql import SparkSession

# Initialize unified SparkSession for local containerized runtime
spark = SparkSession.builder \
    .appName("Predicting Tips - PySpark Linear Regression") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 00:10:42 WARN Utils: Your hostname, MacBook-Air-cua-ang.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.162 instead (on interface en0)
26/10/02 00:10:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/dangthanh/Downloads/learncode/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/02 00:10:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Distributed Data Ingestion & Catalyst Schema Inference

### Code Explanation: Lazy Evaluation & Schema Inference
PySpark DataFrames represent distributed, lazily evaluated collections of structured records divided into logical partitions across cluster resources.

When executing `spark.read.csv()`:
- `header=True`: Designates the first line of the CSV file as column attribute labels.
- `inferSchema=True`: Instructs Catalyst to execute an initial pass over the data partition to automatically infer primitive datatypes (such as distinguishing IEEE double-precision floats from integer counts and strings).
- **Lazy Evaluation**: Spark defers physical disk reading until an *action* (e.g., `.show(5)`) is invoked, allowing Catalyst to construct and optimize the complete physical execution plan.


In [2]:
# Load tips.csv from local container filesystem
df = spark.read.csv("tips.csv", header=True, inferSchema=True)

# Action: materializes first 5 records
df.show(5)


+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
+----------+----+------+------+---+------+----+
only showing top 5 rows


## 3. Schema Reflection & DataType Validation

### Code Explanation: Catalyst DataType Structure
Calling `.printSchema()` reflects the structural metadata tree constructed by Catalyst:
- **Continuous Features**: `total_bill` (`DoubleType`), `tip` (`DoubleType`)
- **Discrete Count Features**: `size` (`IntegerType`)
- **Nominal Categorical Features**: `sex`, `smoker`, `day`, `time` (`StringType`)

Linear regression algorithms operate strictly on continuous real-valued vectors $\mathbf{x} \in \mathbb{R}^d$. Nominal string categories must undergo encoding into numerical indices before vector assembly.


In [3]:
# Print structural schema and verified data types
df.printSchema()


root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)



## 4. Problem Formulation: Target Variable ($y$) vs. Independent Covariates ($\mathbf{x}$)

### Mathematical Formulation: Regression Problem Setup
In the restaurant tipping domain, the business and analytical objective is to predict the gratuity amount based on table attributes:
- **Target / Dependent Variable ($y$)**: `tip` $\in \mathbb{R}^+$ (continuous dollar amount).
- **Independent Feature Space ($\mathbf{x}$)**:
  1. `total_bill`: continuous bill amount before tip.
  2. `size`: discrete party size.
  3. `sex`, `smoker`, `day`, `time`: categorical social, demographic, and temporal features.

$$\mathbf{x} = [x_{\text{total\_bill}}, x_{\text{size}}, x_{\text{sex}}, x_{\text{smoker}}, x_{\text{day}}, x_{\text{time}}]^T \in \mathbb{R}^6$$

$$\hat{y} = f(\mathbf{x}) \approx \text{tip}$$


In [4]:
# Display all available attributes in raw DataFrame
df.columns


['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']

## 5. Categorical Feature Encoding via `StringIndexer`

### Code Explanation: Estimator vs. Transformer Pattern
PySpark MLlib formalizes algorithm composition using two fundamental pipeline primitives:
1. **Estimator**: An algorithm that must be fitted on an input DataFrame to learn parameters from data. It exposes the interface:
   $$\text{Estimator}.\text{fit}(\mathcal{D}) \longrightarrow \text{Transformer (Model)}$$
   `StringIndexer` is an Estimator: it scans the dataset to observe all unique categorical tokens and computes their empirical frequency distribution.
2. **Transformer**: An algorithm that maps one DataFrame to another by appending transformed feature columns:
   $$\text{Transformer}.\text{transform}(\mathcal{D}) \longrightarrow \mathcal{D}'$$
   The fitted `StringIndexerModel` is a Transformer that projects string tokens into deterministic numerical indices.

### Mathematical Formulation: Frequency-Ranked Index Mapping
For an arbitrary categorical feature with distinct categories $\mathcal{C} = \{c_1, c_2, \dots, c_K\}$, `StringIndexer` maps categories to zero-based indices $\{0, 1, \dots, K-1\}$ sorted descending by frequency:

$$\text{Index}(c) = \operatorname{rank}_{\text{desc}}\Big(|\{i : x^{(i)} = c\}|\Big), \quad \text{where } \text{Index}(c_{(0)}) = 0.0 \text{ for } c_{(0)} = \arg\max_{c} \operatorname{count}(c)$$

Using a **single multi-column `StringIndexer` pass** (`inputCols` and `outputCols`) encodes `['sex', 'smoker', 'day', 'time']` simultaneously in a single distributed pass over the dataset, eliminating redundant scans.


In [5]:
from pyspark.ml.feature import StringIndexer

# Configure single multi-column StringIndexer pass
categorical_cols = ["sex", "smoker", "day", "time"]
indexed_cols = ["sex_indexed", "smoker_indexed", "day_indexed", "time_indexed"]

indexer = StringIndexer(inputCols=categorical_cols, outputCols=indexed_cols)

# fit() calculates category frequencies (Estimator -> Model)
# transform() appends indexed columns (Transformer)
df_indexed = indexer.fit(df).transform(df)

# Validate encoded columns against raw categories
df_indexed.select(categorical_cols + indexed_cols).show(5)


+------+------+---+------+-----------+--------------+-----------+------------+
|   sex|smoker|day|  time|sex_indexed|smoker_indexed|day_indexed|time_indexed|
+------+------+---+------+-----------+--------------+-----------+------------+
|Female|    No|Sun|Dinner|        1.0|           0.0|        1.0|         0.0|
|  Male|    No|Sun|Dinner|        0.0|           0.0|        1.0|         0.0|
|  Male|    No|Sun|Dinner|        0.0|           0.0|        1.0|         0.0|
|  Male|    No|Sun|Dinner|        0.0|           0.0|        1.0|         0.0|
|Female|    No|Sun|Dinner|        1.0|           0.0|        1.0|         0.0|
+------+------+---+------+-----------+--------------+-----------+------------+
only showing top 5 rows


## 6. Feature Vectorization via `VectorAssembler`

### Code Explanation: Unified Feature Columns & BLAS Interface
Unlike single-machine frameworks (such as Scikit-Learn) where feature matrices are stored as contiguous 2D NumPy arrays, PySpark MLlib stores each instance as a discrete DataFrame row. To enable distributed linear algebra routines (BLAS / LAPACK) across cluster partitions, all predictor features must be coalesced into a single vector column (`featuresCol`).

`VectorAssembler` is a **Transformer** that merges an ordered list of scalar and categorical index columns into a unified feature vector:
$$\mathbf{x} = [x_1, x_2, \dots, x_d]^T$$

### Mathematical Formulation: DenseVector vs. SparseVector
PySpark MLlib supports two vector storage implementations:
1. **DenseVector**: Stores all scalar values explicitly in a contiguous array of IEEE 64-bit floating-point numbers:
   $$\mathbf{v}_{\text{dense}} = [v_0, v_1, \dots, v_{d-1}]$$
2. **SparseVector**: Stores vector dimension $d$, an array of non-zero integer indices, and an array of non-zero double values:
   $$\mathbf{v}_{\text{sparse}} = \big(d, [i_0, i_1, \dots, i_{k-1}], [v_{i_0}, v_{i_1}, \dots, v_{i_{k-1}}]\big)$$

*Sparsity Optimization*: When categorical variables are indexed, the most frequent level maps to index $0.0$. When multiple components equal $0.0$, `VectorAssembler` automatically switches between dense and sparse representations to minimize memory consumption and network serialization overhead during shuffle operations.


In [6]:
from pyspark.ml.feature import VectorAssembler

# Independent features: total_bill MUST be included as an independent predictor
feature_cols = ['total_bill', 'size', 'sex_indexed', 'smoker_indexed', 'day_indexed', 'time_indexed']

assembler = VectorAssembler(inputCols=feature_cols, outputCol="Independent Features")
df_assembled = assembler.transform(df_indexed)

# Show assembled vectors alongside the target label
df_assembled.select("Independent Features", "tip").show(5, truncate=False)


+---------------------------+----+
|Independent Features       |tip |
+---------------------------+----+
|[16.99,2.0,1.0,0.0,1.0,0.0]|1.01|
|[10.34,3.0,0.0,0.0,1.0,0.0]|1.66|
|[21.01,3.0,0.0,0.0,1.0,0.0]|3.5 |
|[23.68,2.0,0.0,0.0,1.0,0.0]|3.31|
|[24.59,4.0,1.0,0.0,1.0,0.0]|3.61|
+---------------------------+----+
only showing top 5 rows


## 7. Dataset Finalization & Memory Optimization

### Code Explanation: Column Pruning & Partition Optimization
In distributed query execution, serializing unused columns across iterative optimization stages incurs significant network and memory overhead. We isolate exclusively:
- Feature vector: `Independent Features` (Design matrix $\mathbf{X}$)
- Target label: `tip` (Response vector $\mathbf{y}$)


In [7]:
# Select only the assembled predictor vector and the target label
finalized_data = df_assembled.select("Independent Features", "tip")
finalized_data.show(5, truncate=False)


+---------------------------+----+
|Independent Features       |tip |
+---------------------------+----+
|[16.99,2.0,1.0,0.0,1.0,0.0]|1.01|
|[10.34,3.0,0.0,0.0,1.0,0.0]|1.66|
|[21.01,3.0,0.0,0.0,1.0,0.0]|3.5 |
|[23.68,2.0,0.0,0.0,1.0,0.0]|3.31|
|[24.59,4.0,1.0,0.0,1.0,0.0]|3.61|
+---------------------------+----+
only showing top 5 rows


## 8. Distributed Train-Test Partitioning with Seed Reproducibility

### Code Explanation: Bernoulli Sampling Across Partitions
PySpark's `.randomSplit([0.75, 0.25], seed=42)` partitions the DataFrame into:
- **Training Subset ($75\%$)**: Used for computing parameter estimates $(\mathbf{w}, b)$.
- **Testing Subset ($25\%$)**: Unseen out-of-sample data used for unbiased generalization evaluation.

Under the hood, `randomSplit` computes a pseudo-random hash value for each row using the designated `seed=42`. Enforcing an explicit random seed ensures deterministic row assignment across executors and prevents data leakage across successive experimental runs.


In [8]:
# Partition dataset into train and test splits with fixed seed
train_data, test_data = finalized_data.randomSplit([0.75, 0.25], seed=42)

print(f"Training dataset row count : {train_data.count()}")
print(f"Testing dataset row count  : {test_data.count()}")


Training dataset row count : 195
Testing dataset row count  : 49


## 9. Multi-Variable Ordinary Least Squares (OLS) Linear Regression

### Mathematical Hypothesis
For each observation $i$, given feature vector $\mathbf{x}^{(i)} = [x_1^{(i)}, x_2^{(i)}, \dots, x_d^{(i)}]^T$, the linear regression hypothesis estimates predicted tip $\hat{y}^{(i)}$ as:

$$\hat{y}^{(i)} = \mathbf{w}^T \mathbf{x}^{(i)} + b = b + \sum_{j=1}^{d} w_j x_j^{(i)}$$

where $\mathbf{w} \in \mathbb{R}^d$ is the weight vector and $b \in \mathbb{R}$ is the intercept term.

### Ordinary Least Squares (OLS) Objective Function
The optimal model parameters $\mathbf{w}^*$ and $b^*$ minimize the empirical Mean Squared Error loss across $N$ training instances:

$$J(\mathbf{w}, b) = \frac{1}{2N} \sum_{i=1}^{N} \left( y^{(i)} - (\mathbf{w}^T \mathbf{x}^{(i)} + b) \right)^2 = \frac{1}{2N} \|\mathbf{y} - (\mathbf{X}\mathbf{w} + b\mathbf{1})\|_2^2$$

### Partial Derivatives & Optimization
The analytical gradients of the cost function are:

$$\nabla_{\mathbf{w}} J(\mathbf{w}, b) = -\frac{1}{N} \mathbf{X}^T \left( \mathbf{y} - (\mathbf{X}\mathbf{w} + b\mathbf{1}) \right)$$

$$\frac{\partial J(\mathbf{w}, b)}{\partial b} = -\frac{1}{N} \sum_{i=1}^{N} \left( y^{(i)} - (\mathbf{w}^T \mathbf{x}^{(i)} + b) \right)$$

Setting $\nabla J = \mathbf{0}$ yields the analytical Normal Equation:
$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In PySpark MLlib, `LinearRegression` (an **Estimator**) computes these parameters distributedly using Cholesky factorization of $\mathbf{X}^T \mathbf{X}$ or quasi-Newton numerical solvers (L-BFGS).


In [9]:
from pyspark.ml.regression import LinearRegression

# Formulate Linear Regression with featuresCol='Independent Features' and labelCol='tip'
lr = LinearRegression(featuresCol='Independent Features', labelCol='tip')

# Fit Estimator to training partition to obtain trained LinearRegressionModel (Transformer)
regressor = lr.fit(train_data)


26/10/02 00:10:51 WARN Instrumentation: [b39e8302] regParam is zero, which might cause numerical instability and overfitting.


## 10. Model Parameter Extraction & Mathematical Interpretation

### Mathematical Meaning of Weights & Bias
- **Intercept ($b$)**: The baseline expected tip ($) when all independent feature values are zero.
- **Partial Slopes ($w_j = \frac{\partial \hat{y}}{\partial x_j}$)**: The expected marginal change in tip for each unit increase in feature $x_j$, holding all other covariates constant (ceteris paribus). For example, $w_{\text{total\_bill}}$ represents the expected additional tip per dollar of bill.


In [10]:
# Display scalar intercept term b
print(f"Model Intercept (b): {regressor.intercept:.6f}\n")

# Display feature weights vector w
print("Model Coefficients (w):")
for feature_name, coef in zip(feature_cols, regressor.coefficients):
    print(f"  - {feature_name:15s}: {coef:+.6f}")


Model Intercept (b): 0.642239

Model Coefficients (w):
  - total_bill     : +0.104809
  - size           : +0.117434
  - sex_indexed    : +0.095518
  - smoker_indexed : -0.110002
  - day_indexed    : -0.005682
  - time_indexed   : -0.034359


## 11. Out-of-Sample Batch Distributed Inference

### Code Explanation: Model Evaluation & Residual Analysis
Calling `regressor.evaluate(test_data)` executes distributed batch inference on the unseen test partition. The trained model computes:

$$\hat{\mathbf{y}}_{\text{test}} = \mathbf{X}_{\text{test}} \mathbf{w} + b\mathbf{1}$$

The resulting `LinearRegressionSummary` yields DataFrame `pred_results.predictions` containing:
- `Independent Features`: Input vector $\mathbf{x}^{(i)}$
- `tip`: Ground-truth observed label $y^{(i)}$
- `prediction`: Fitted prediction $\hat{y}^{(i)}$
- Residual error: $e^{(i)} = y^{(i)} - \hat{y}^{(i)}$


In [11]:
# Evaluate model performance on out-of-sample test partition
pred_results = regressor.evaluate(test_data)

# Display predictions side-by-side with ground-truth labels
pred_results.predictions.select("Independent Features", "tip", "prediction").show(10, truncate=False)


+--------------------------+----+------------------+
|Independent Features      |tip |prediction        |
+--------------------------+----+------------------+
|(6,[0,1],[10.51,2.0])     |1.25|1.9786488350745728|
|(6,[0,1],[12.69,2.0])     |2.0 |2.2071321185025647|
|(6,[0,1],[13.37,2.0])     |2.0 |2.278402133516801 |
|(6,[0,1],[14.0,2.0])      |3.0 |2.3444317062505786|
|(6,[0,1],[17.78,2.0])     |3.27|2.7406091426532444|
|(6,[0,1],[18.69,3.0])     |2.31|2.9534194445972908|
|(6,[0,1],[20.45,4.0])     |3.0 |3.255317265309132 |
|(6,[0,1],[39.42,4.0])     |7.58|5.243541066515102 |
|(6,[0,1],[48.27,4.0])     |6.73|6.171099350156264 |
|[3.07,1.0,1.0,1.0,0.0,0.0]|1.0 |1.0669529857467526|
+--------------------------+----+------------------+
only showing top 10 rows


## 12. Quantitative Evaluation Metrics Formulation

To rigorously assess out-of-sample predictive accuracy, we compute four standardized regression metrics:

### 1. Mean Absolute Error (MAE)
Quantifies the average magnitude of absolute errors, offering robustness against extreme outliers:
$$\text{MAE} = \frac{1}{N} \sum_{i=1}^{N} |y^{(i)} - \hat{y}^{(i)}|$$

### 2. Mean Squared Error (MSE)
Penalizes larger prediction errors quadratically, reflecting expected variance of residuals:
$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y^{(i)} - \hat{y}^{(i)})^2$$

### 3. Root Mean Squared Error (RMSE)
Scale-dependent error metric represented in the native unit of the target variable (dollars):
$$\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y^{(i)} - \hat{y}^{(i)})^2}$$

### 4. Coefficient of Determination ($R^2$)
Measures the proportion of total variance in the dependent variable explained by the linear regression model relative to a naive mean baseline:
$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum_{i=1}^{N} (y^{(i)} - \hat{y}^{(i)})^2}{\sum_{i=1}^{N} (y^{(i)} - \bar{y})^2}$$

where $\bar{y} = \frac{1}{N} \sum_{i=1}^{N} y^{(i)}$ is the empirical sample mean of the target variable.


In [12]:
# Retrieve evaluation metrics from LinearRegressionSummary
mae = pred_results.meanAbsoluteError
mse = pred_results.meanSquaredError
rmse = pred_results.rootMeanSquaredError
r2 = pred_results.r2

print("=" * 48)
print("       TEST SET REGRESSION EVALUATION METRICS   ")
print("=" * 48)
print(f"  Mean Absolute Error (MAE)        : {mae:.4f}")
print(f"  Mean Squared Error (MSE)         : {mse:.4f}")
print(f"  Root Mean Squared Error (RMSE)   : {rmse:.4f}")
print(f"  Coefficient of Determination (R2): {r2:.4f}")
print("=" * 48)


       TEST SET REGRESSION EVALUATION METRICS   
  Mean Absolute Error (MAE)        : 0.8617
  Mean Squared Error (MSE)         : 1.5180
  Root Mean Squared Error (RMSE)   : 1.2321
  Coefficient of Determination (R2): 0.2937


## 13. Resource Management & SparkSession Teardown

In shared container and distributed cluster environments, invoking `spark.stop()` ensures graceful termination of the execution context:
- Releases allocated executor JVM memory and CPU worker threads.
- Closes Py4J gateway socket connections.
- Prevents port collision (such as Spark Web UI default port 4040) upon subsequent invocations.


In [13]:
# Gracefully stop the SparkSession
spark.stop()
print("SparkSession successfully stopped.")


SparkSession successfully stopped.
